<!-- TARGET: public course-materials repo, sessions/m07/demo.ipynb (full version, after class). Copy to work/m07/ before editing. -->
# Module 7 · Live demo

**Python for Data Science · ESADE MIBA · Wed 25 Nov 2026, 9:00**

Last week: a table, columns mapped to channels, the chart that fits the question, and a check
that the chart shows the numbers we think. Today: charts that **mislead** and how to fix them,
**maps**, **time series**, and a chart check you will hand in (CP-viz).

Every chart starts from the **order mart** (`data/mart/orders_enriched.parquet`) and, for maps,
the state outlines in `data/geo/brazil-states.geojson`. If the cell below stops with an
`AssertionError`, copy `sessions/m05/lab-solution.ipynb` to `work/m05/`, Run All, and come back.
The cell also sets the ESADE chart style from M06 and keeps `delivered`: delivered orders with a
delivery date.

In [ ]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

RAW = Path("../../data/raw")
MART_FILE = Path("../../data/mart/orders_enriched.parquet")
GEO_FILE = Path("../../data/geo/brazil-states.geojson")

assert MART_FILE.exists(), (
    "No order mart yet. Copy sessions/m05/lab-solution.ipynb to work/m05/, Run All, then come back."
)
assert GEO_FILE.exists(), "No map file. Pull the course repository: it ships data/geo/brazil-states.geojson."
mart = pd.read_parquet(MART_FILE)
assert mart.shape == (99441, 13), (
    f"Your mart has shape {mart.shape}, expected (99441, 13). Run sessions/m05/lab-solution.ipynb to rebuild it."
)
with open(GEO_FILE) as geo_file:
    brazil_states = json.load(geo_file)

ESADE_BLUE = "#00205B"
ESADE_LIGHT_BLUE = "#4A9EFF"
GREY = "#B8BEC8"
pio.templates["esade"] = go.layout.Template(
    layout=dict(
        font=dict(family="Arial", color="#2B2B2B"),
        title=dict(font=dict(color=ESADE_BLUE, size=20)),
        colorway=[ESADE_BLUE, ESADE_LIGHT_BLUE, "#E07A1F", "#7A869A", "#2CA58D", "#C1272D"],
    )
)
pio.templates.default = "plotly_white+esade"

delivered = mart[(mart["order_status"] == "delivered") & mart["is_late"].notna()]

## Hook · Three charts from a slide deck

A colleague's deck for the board has three charts. Each one is built from correct numbers. *What
is wrong with each?*

In [ ]:
monthly = (
    mart.groupby("purchase_month")
    .agg(orders=("order_id", "size"), mean_score=("review_score", "mean"))
    .reset_index()
)
in_range = monthly[monthly["purchase_month"].between("2017-01", "2018-08")]
fig_score_drama = px.bar(in_range, x="purchase_month", y="mean_score", title="Customer satisfaction collapsed in early 2018")
fig_score_drama.update_layout(yaxis_range=[3.7, 4.3])
fig_score_drama

In [ ]:
items = pd.read_csv(RAW / "olist_order_items_dataset.csv")
products = pd.read_csv(RAW / "olist_products_dataset.csv")
translation = pd.read_csv(RAW / "product_category_name_translation.csv")

items_with_category = items.merge(
    products[["product_id", "product_category_name"]], on="product_id", how="left", validate="many_to_one"
).merge(translation, on="product_category_name", how="left", validate="many_to_one")
items_with_category = items_with_category.assign(
    category=items_with_category["product_category_name_english"]
    .fillna(items_with_category["product_category_name"])
    .fillna("unknown"),
    item_revenue=items_with_category["price"] + items_with_category["freight_value"],
)
by_category = (
    items_with_category.groupby("category")
    .agg(item_revenue=("item_revenue", "sum"))
    .sort_values("item_revenue", ascending=False)
    .reset_index()
)
px.pie(by_category, names="category", values="item_revenue", title="Where our revenue comes from")

In [ ]:
px.line(monthly, x="purchase_month", y="orders", title="Orders collapsed in September 2018")

1. **Truncated axis.** The bars start at 3.7. A bar's *length* is its value, so a drop from
   4.04 to 3.75 (7%) looks like a fall of more than 80%.
2. **Pie of 74 slices.** Nobody can compare angles, and nobody can read 74 colours. Which is
   bigger: `watches_gifts` or `bed_bath_table`?
3. **Truncated data.** The extract ends in October 2018: September has 16 orders, October 4.
   The business didn't collapse; the data stops.

The numbers were correct each time. **A chart can mislead with correct numbers.**

## 1. Critique and fix

| Question | Chart | Watch out for |
|---|---|---|
| Compare groups | bar, **sorted**; horizontal if the labels are long | bars start at zero |
| Parts of a whole | sorted bar; a pie only with 2–4 parts | "other" for the long tail |
| Change over time | line | incomplete first and last periods |
| Two measures over time | two panels, one x axis | dual y axes |
| Where | map, **normalised** | totals just show where people live |

### Fix 1: the truncated axis

A line is read by its *shape*, not its length, so a line may zoom in. Use the score's real
scale (1 to 5), and say what you show.

In [ ]:
fig_score = px.line(
    in_range,
    x="purchase_month",
    y="mean_score",
    title="Mean review score dipped in Feb–Mar 2018, the months with the most late deliveries",
    labels={"purchase_month": "Purchase month", "mean_score": "Mean review score (1–5)"},
)
fig_score.update_layout(yaxis_range=[1, 5])
fig_score

The dip is real (4.04 in January, 3.75 in March), and M06 showed why: March 2018 had 18% late
deliveries. Fixing the chart didn't hide the finding. It showed its true size.

### Fix 2: the pie

Ten categories make 62% of item revenue. Show those, sorted, and put the long tail in one
`"other"` bar.

*Your turn (3 min):* build `top_categories`: the ten largest rows of `by_category`, plus one row
`{"category": "other", "item_revenue": <the rest>}`. `pd.concat` stacks two tables.

In [ ]:
top_ten = by_category.head(10)
other = pd.DataFrame({"category": ["other"], "item_revenue": [by_category["item_revenue"].iloc[10:].sum()]})
top_categories = pd.concat([top_ten, other], ignore_index=True)

In [ ]:
top_ten = by_category.head(10)
other = pd.DataFrame({"category": ["other"], "item_revenue": [by_category["item_revenue"].iloc[10:].sum()]})
top_categories = pd.concat([top_ten, other], ignore_index=True)
assert abs(top_categories["item_revenue"].sum() - by_category["item_revenue"].sum()) < 0.01

fig_categories = px.bar(
    top_categories,
    x="item_revenue",
    y="category",
    orientation="h",
    title="Ten categories bring 62% of item revenue; health & beauty leads",
    labels={"item_revenue": "Item revenue (BRL)", "category": ""},
)
fig_categories.update_layout(yaxis_autorange="reversed")
fig_categories

Horizontal bars leave room for long labels. `yaxis_autorange="reversed"` puts the first row at
the top. The `assert` proves that no revenue went missing on the way.

### Fix 3: the truncated data

Cut the months the extract doesn't cover, and say so. Last week's demo did exactly that.

In [ ]:
px.line(
    in_range,
    x="purchase_month",
    y="orders",
    title="Orders grew through 2017 and levelled off in 2018 (Jan 2017 – Aug 2018: complete months only)",
    labels={"purchase_month": "Purchase month", "orders": "Orders"},
)

### Two measures over time: panels, not two y axes

A board member asks: *did late deliveries grow with volume?* A dual-axis chart puts orders on the
left and late share on the right. You choose both scales, so you choose where the lines cross,
and the reader sees a relationship you drew. Two panels that share the x axis are honest: one
measure, one axis.

In [ ]:
late_by_month = (
    delivered[delivered["purchase_month"].between("2017-01", "2018-08")]
    .groupby("purchase_month")
    .agg(orders=("order_id", "size"), late_share=("is_late", "mean"))
    .reset_index()
)
late_long = late_by_month.melt(id_vars="purchase_month", var_name="measure", value_name="value")
fig_panels = px.line(
    late_long,
    x="purchase_month",
    y="value",
    facet_row="measure",
    color="measure",
    title="Volume peaked in Nov 2017; lateness peaked in Mar 2018",
    labels={"purchase_month": "Purchase month", "value": ""},
)
fig_panels.update_yaxes(matches=None)
fig_panels

`facet_row` stacks the panels; `update_yaxes(matches=None)` gives each panel its own y scale.
November 2017 had the most orders; March 2018 had the most lateness. Not the same month.

### Declutter: grey, plus one colour

An executive needs one message per chart. Colour only what the title talks about.

In [ ]:
NORTH_EAST = ["AL", "BA", "CE", "MA", "PB", "PE", "PI", "RN", "SE"]
late_by_state = (
    delivered.groupby("customer_state")
    .agg(orders=("order_id", "size"), late_orders=("is_late", "sum"), late_share=("is_late", "mean"))
    .reset_index()
    .sort_values("late_share", ascending=False)
)
late_by_state["region"] = late_by_state["customer_state"].isin(NORTH_EAST).map({True: "North-East", False: "rest of Brazil"})
fig_focus = px.bar(
    late_by_state,
    x="customer_state",
    y="late_share",
    color="region",
    color_discrete_map={"North-East": ESADE_BLUE, "rest of Brazil": GREY},
    title="Six of the seven states with the most late deliveries are in the North-East",
    labels={"customer_state": "Customer state", "late_share": "Share delivered late", "region": ""},
)
fig_focus.update_layout(yaxis_tickformat=".0%")
fig_focus

## 2. Maps

Same table, drawn on Brazil. `px.choropleth` needs three things:

- `geojson`: the shapes (here, 27 states);
- `locations`: the column holding each row's state code;
- `featureidkey`: where that code lives inside each shape. Look at one:

In [ ]:
brazil_states["features"][0]["properties"]

The two-letter code is in `properties.sigla`, the same codes as `customer_state`. First, the
**count** of late orders:

In [ ]:
fig_late_count_map = px.choropleth(
    late_by_state,
    geojson=brazil_states,
    locations="customer_state",
    featureidkey="properties.sigla",
    color="late_orders",
    color_continuous_scale="Reds",
    title="Late orders (count)",
)
fig_late_count_map.update_geos(fitbounds="locations", visible=False)
fig_late_count_map

São Paulo lights up. That's a population map: SP has 42% of the orders. Totals on a map show
where the customers are. To show **risk**, normalise: late orders **per order**.

*Your turn (3 min):* the same map, coloured by `late_share`. Store it in `fig_late_map`, with a
title that says the finding.

In [ ]:
fig_late_map = px.choropleth(
    late_by_state,
    geojson=brazil_states,
    locations="customer_state",
    featureidkey="properties.sigla",
    color="late_share",
    color_continuous_scale="Reds",
    hover_data=["orders", "late_orders"],
    title="The North-East has the highest share of late deliveries",
    labels={"late_share": "Share late"},
)
fig_late_map.update_geos(fitbounds="locations", visible=False)
fig_late_map

### Colour scales

- **Sequential** (`"Reds"`, `"Blues"`): low → high. The default choice.
- **Diverging** (`"RdBu_r"`): two directions from a meaningful middle. Here: the national late
  share. Red is worse than Brazil overall, blue is better.
- Avoid rainbow scales: the eye sees borders between hues that aren't in the data.

In [ ]:
national_late_share = delivered["is_late"].mean()
fig_vs_national = px.choropleth(
    late_by_state,
    geojson=brazil_states,
    locations="customer_state",
    featureidkey="properties.sigla",
    color="late_share",
    color_continuous_scale="RdBu_r",
    color_continuous_midpoint=national_late_share,
    title=f"Red: later than Brazil overall ({national_late_share:.1%}). The North-East and Rio de Janeiro stand out",
    labels={"late_share": "Share late"},
)
fig_vs_national.update_geos(fitbounds="locations", visible=False)
fig_vs_national

## 3. Time series

Months hide what happens inside them. To count orders by **week**, put the timestamp in the index
and `resample`: it is a `groupby` on calendar periods. `"W"` labels each week by its Sunday.

In [ ]:
weekly = (
    mart.set_index("order_purchase_timestamp")
    .resample("W")
    .agg(orders=("order_id", "size"))
)
px.line(weekly.reset_index(), x="order_purchase_timestamp", y="orders")

Both ends are thin. At the end, the last complete-looking week (ending 26 Aug 2018) has 1,071
orders against about 1,900 the weeks before: the extract tails off **inside** August. Keep the
weeks from 8 Jan 2017 to 19 Aug 2018. A `DatetimeIndex` slices with dates as strings.

In [ ]:
weekly = weekly.loc["2017-01-08":"2018-08-19"]
print(len(weekly), "weeks")
weekly.loc["2017-11-12":"2017-12-03"]

Black Friday week (ending Sun 26 Nov 2017): **3,008** orders, more than twice the week before.

Weekly data is noisy. A **rolling mean** averages each week with the weeks before it:
`rolling(4).mean()` is the mean of the last four weeks.

*Your turn (2 min):* add a column `rolling_4w` to `weekly` with the 4-week rolling mean of
`orders`. *Predict:* how many of its values are missing, and why?

In [ ]:
weekly["rolling_4w"] = weekly["orders"].rolling(4).mean()

In [ ]:
weekly["rolling_4w"] = weekly["orders"].rolling(4).mean()
print(weekly["rolling_4w"].isna().sum(), "missing: the first three weeks have no four weeks behind them")

weekly_long = weekly.reset_index().melt(
    id_vars="order_purchase_timestamp", value_vars=["orders", "rolling_4w"], var_name="series", value_name="value"
)
fig_weekly = px.line(
    weekly_long,
    x="order_purchase_timestamp",
    y="value",
    color="series",
    title="Black Friday 2017 doubled a week's orders; the 4-week mean shows the trend",
    labels={"order_purchase_timestamp": "Week (ending Sunday)", "value": "Orders", "series": ""},
)
fig_weekly

## 4. Chart = table, again

The running question: **how do you know this number is right?** A map is a table too. A bar
chart keeps its numbers in `fig.data[0].x` and `.y`; a choropleth keeps them in
`fig.data[0].locations` and `fig.data[0].z`.

In [ ]:
plotted = pd.Series(list(fig_vs_national.data[0].z), index=list(fig_vs_national.data[0].locations))
expected = delivered.groupby("customer_state")["is_late"].mean()

assert len(plotted) == 27, "one value per state"
assert ((plotted - expected).abs() < 1e-9).all(), "the map shows the late share of every state"
plotted.sort_values(ascending=False).head()

*Your turn (2 min):* check `fig_weekly`. Its first trace (`fig_weekly.data[0]`) is the weekly
orders. `assert` that its values add up to the number of orders in `mart` placed between 2 Jan
2017 and 19 Aug 2018 (inclusive).

In [ ]:
in_window = mart["order_purchase_timestamp"].between("2017-01-02", "2018-08-20", inclusive="left")
assert sum(fig_weekly.data[0].y) == in_window.sum()

### Share it

`write_html` saves a chart as a web page that anyone can open in a browser, with the zoom and the
tooltips, and without Python. Check that the file exists.

In [ ]:
html_file = Path("late-share-map.html")
fig_vs_national.write_html(html_file, include_plotlyjs="cdn")
assert html_file.exists()
print(html_file.resolve())

`include_plotlyjs="cdn"` keeps the file small; the browser loads plotly from the internet. Leave
it out for a file that works offline (about 4 MB).

## Today in one table

| Situation | Do |
|---|---|
| Bars | start at zero; sort them |
| Lines | may zoom; say what range you show |
| Many parts of a whole | sorted bar, top N + "other"; a pie only with 2–4 parts |
| First or last period incomplete | cut it, and say so |
| Two measures over time | two panels (`facet_row`, `update_yaxes(matches=None)`), not two y axes |
| One message | grey, plus one colour |
| Map | `geojson`, `locations`, `featureidkey`; normalise per order; sequential scale, diverging only around a meaningful middle |
| Weekly | `set_index(timestamp).resample("W")`, `rolling(4).mean()` |
| Check | bars: `.x`, `.y`; maps: `.locations`, `.z`; assert against a fresh `groupby`; `write_html` to share |